# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [61]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr


In [62]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [63]:
# Load tables
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')

In [64]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{' g__Janibacter', ' g__Reyranella', ' g__Mycobacterium'}

Filtered Unique to V4 (>=10% prevalence):
{' g__Brachybacterium', ' g__Fenollaria', ' g__Aeromonas', ' g__Pantoea', ' g__Chryseobacterium', ' g__Gardnerella', ' g__Alloiococcus', ' g__Turicella', ' g__Bifidobacterium', ' g__Empedobacter', ' g__Jeotgalicoccus', ' g__Aggregatibacter', ' g__Fusobacterium', ' g__Peptoniphilus', ' g__Psychrobacter', ' g__Aerococcus', ' g__Geobacillus', ' g__Atopobium', ' g__Frederiksenia', ' g__Bradyrhizobium', ' g__Capnocytophaga', ' g__Enhydrobacter', ' g__Dolosigranulum', ' g__Blautia', ' g__Finegoldia', ' g__Stenotrophomonas', ' g__Abiotrophia', ' g__Lactococcus', ' g__Massilia', ' g__Leptotrichia', ' g__Moraxella', ' g__Leuconostoc', ' g__Marinomonas', ' g__Peptostreptococcus', ' g__Vibrio', ' g__Luteimonas'}

Filtered Shared Taxa (>=10% prevalence):
{' g__Actinomyces', ' g__Lactobacillus', ' g__Pseudomonas', ' g__Corynebacterium', ' g__Caulobacter', 

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_19576/4290181383.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [65]:
# Read and transform V1-V3 and V4 Genus level absolute abundance table
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed.biom')

In [66]:
# Extract the row corresponding to 'g__Cutibacterium' for V1-V3
V1_V3_staphylococcus_df = V1V3_biom.loc[[' g__Staphylococcus']]

# Rename the row index
V1_V3_staphylococcus_df.index = [' g__Staphylococus V1-V3']

# Transpose the DataFrame
V1_V3_staphylococcus_df = V1_V3_staphylococcus_df.T

# Display the transformed DataFrame
V1_V3_staphylococcus_df

,g__Staphylococus V1-V3
LAMI.RD001.D0.C1,0.024940
LAMI.RD001.D0.C2,0.041921
LAMI.RD001.D14.C1,0.022022
LAMI.RD001.D14.C2,0.036349
LAMI.RD001.D28.C1,0.031308
...,...
LAMI.RD319.D14.C1,0.002123
LAMI.RD319.D21.C3,0.002919
LAMI.RD319.D14.C2,0.002123
LAMI.RD319.D9.C1,0.003449


In [67]:
# Extract the row corresponding to 'g__Cutibacterium' for V4
V4_staphylococcus_df = V4_biom.loc[[' g__Staphylococcus']]

# Rename the row index
V4_staphylococcus_df.index = [' g__Staphyloccocus V4']

# Transpose the DataFrame
V4_staphylococcus_df = V4_staphylococcus_df.T

# Display the transformed DataFrame
V4_staphylococcus_df

,g__Staphyloccocus V4
LAMI.RD001.D0.C1,0.084107
LAMI.RD001.D14.C1,0.120456
LAMI.RD004.D0.C2,0.438313
LAMI.RD001.D0.C2,0.092598
LAMI.RD004.D28.C1,0.088087
...,...
LAMI.RD319.D16.C2,0.037676
LAMI.RD319.D28.C1,0.011144
LAMI.RD318.D9.C2,0.036880
LAMI.RD319.D28.C2,0.013797


In [68]:
# Get ranks for V1-V3
V1_V3_staphylococcus_df['V1-V3'] = V1_V3_staphylococcus_df[' g__Staphylococus V1-V3'].rank(method='min').astype(int)
V1_V3_staphylococcus_df

,g__Staphylococus V1-V3,V1-V3
LAMI.RD001.D0.C1,0.024940,172
LAMI.RD001.D0.C2,0.041921,201
LAMI.RD001.D14.C1,0.022022,168
LAMI.RD001.D14.C2,0.036349,194
LAMI.RD001.D28.C1,0.031308,185
...,...,...
LAMI.RD319.D14.C1,0.002123,20
LAMI.RD319.D21.C3,0.002919,38
LAMI.RD319.D14.C2,0.002123,20
LAMI.RD319.D9.C1,0.003449,50


In [69]:
# Get ranks for V4
V4_staphylococcus_df['V4'] = V4_staphylococcus_df[' g__Staphyloccocus V4'].rank(method='min').astype(int)
V4_staphylococcus_df

,g__Staphyloccocus V4,V4
LAMI.RD001.D0.C1,0.084107,44
LAMI.RD001.D14.C1,0.120456,60
LAMI.RD004.D0.C2,0.438313,149
LAMI.RD001.D0.C2,0.092598,49
LAMI.RD004.D28.C1,0.088087,46
...,...,...
LAMI.RD319.D16.C2,0.037676,34
LAMI.RD319.D28.C1,0.011144,11
LAMI.RD318.D9.C2,0.036880,33
LAMI.RD319.D28.C2,0.013797,15


In [70]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
v1_v3_column = V1_V3_staphylococcus_df['V1-V3']  # Adjust column name if needed
v4_column = V4_staphylococcus_df['V4']  # Adjust column name if needed

# Concatenate the selected columns
combined_staphylococcus_df = pd.concat([v1_v3_column, v4_column], axis=1)

# Rename columns for clarity (optional)
combined_staphylococcus_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_staphylococcus_df = combined_staphylococcus_df.dropna()

combined_staphylococcus_df

,V1-V3,V4
LAMI.RD001.D0.C1,172.0,44.0
LAMI.RD001.D0.C2,201.0,49.0
LAMI.RD001.D14.C1,168.0,60.0
LAMI.RD001.D14.C2,194.0,58.0
LAMI.RD001.D28.C1,185.0,67.0
...,...,...
LAMI.RD318.D28.C3,11.0,21.0
LAMI.RD319.D14.C1,20.0,4.0
LAMI.RD319.D14.C2,20.0,22.0
LAMI.RD319.D9.C1,50.0,2.0


In [71]:
# Scatter plot
plt.figure(figsize=(5, 6))
plt.scatter(
    combined_staphylococcus_df.iloc[:, 0], 
    combined_staphylococcus_df.iloc[:, 1], 
    color='#92f0f0', 
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7, 
    label='Samples'
)

# Extract x and y data
x = combined_staphylococcus_df.iloc[:, 0]
y = combined_staphylococcus_df.iloc[:, 1]

# Compute the best-fit line
m, b = np.polyfit(x, y, 1)

# Compute Pearson correlation coefficient and p-value
r, p_value = pearsonr(x, y)

# Plot the best-fit line with the correlation and p-value in the label
plt.plot(x, m * x + b, color='#5fc4c4', label=f'Pearson Correlation: {r:.2f}\np-value: {p_value:.3g}')

# Title and labels
plt.title('Staphylococcus by Primer Set', fontsize=16)
plt.xlabel('Ranked Relative Abundance (V1-V3)', fontsize=14)
plt.ylabel('Ranked Relative Abundance (V4)', fontsize=14)

# Grid, legend, and save
plt.grid(True)
plt.legend()
plt.savefig('../Figures/16S_Figures/primer_comparison/Staphylococcus_V1V3_vs_V4_correlation_ranks.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_19576/4128559969.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Comparison of 16S V1-V3, 16S V4, and shotgun of Cutibacterium per-sample ranked relative abundances with intersectional samples

In [72]:
# Load tables
veba_slcs = pd.read_csv('../Data/metaG/VEBA_analysis/output_files/X_mags.with_taxonomy.with_slcs_name.csv')
veba_slcs_subset = veba_slcs.iloc[:, 4:]
veba_slcs_subset = veba_slcs_subset.set_index('name')
veba_slcs_subset.index.name = None

veba_slcs_subset_collapsed = veba_slcs_subset.groupby(veba_slcs_subset.index).sum()
relative_abundance_df = veba_slcs_subset_collapsed.div(veba_slcs_subset_collapsed.sum(axis=0), axis=1)
relative_abundance_df = relative_abundance_df.T
staphylococcus_only = relative_abundance_df[['Staphylococcus epidermidis']]
staphylococcus_only = staphylococcus_only.rename(columns={'Staphylococcus epidermidis': 'Staphylococcus epidermidis Shotgun'})

staphylococcus_only.index = staphylococcus_only.index.str.split('_').str[:4].str.join('_')

staphylococcus_only.index = staphylococcus_only.index.str.replace('_', '.', regex=False)


# Get ranks for shotgun
staphylococcus_only['Shotgun'] = staphylococcus_only['Staphylococcus epidermidis Shotgun'].rank(method='min').astype(int)
staphylococcus_only

KeyError: "None of [Index(['Staphylococcus epidermidis'], dtype='object')] are in the [columns]"

In [ ]:
# Get ranks for V1-V3 for samples matching with shotgun
V1_V3_staphylococcus_df_shotgun_subset = V1_V3_staphylococcus_df.loc[V1_V3_staphylococcus_df.index.isin(staphylococcus_only.index)]
V1_V3_staphylococcus_df_shotgun_subset = V1_V3_staphylococcus_df_shotgun_subset[[' g__Staphylococus V1-V3']]

# Get ranks for subsetted V1-V3
V1_V3_staphylococcus_df_shotgun_subset['V1-V3'] = V1_V3_staphylococcus_df_shotgun_subset[' g__Staphylococus V1-V3'].rank(method='min').astype(int)
V1_V3_staphylococcus_df_shotgun_subset

,g__Staphylococus V1-V3,V1-V3
LAMI.RD304.D14.C3,0.055187,19
LAMI.RD304.D7.C1,0.060493,20
LAMI.RD304.D0.C1,0.071106,22
LAMI.RD306.D0.C3,0.022552,13
LAMI.RD306.D11.C1,0.037410,16
LAMI.RD308.D0.C2,0.000531,1
LAMI.RD306.D14.C1,0.065800,21
LAMI.RD304.D0.C3,0.079597,23
LAMI.RD308.D0.C3,0.002653,4
LAMI.RD306.D23.C1,0.011674,12


In [ ]:
# Get ranks for V4 for samples matching with shotgun
V4_staphylococcus_df_shotgun_subset = V4_staphylococcus_df.loc[V4_staphylococcus_df.index.isin(staphylococcus_only.index)]
V4_staphylococcus_df_shotgun_subset = V4_staphylococcus_df_shotgun_subset[[' g__Staphyloccocus V4']]

# Get ranks for subsetted V4
V4_staphylococcus_df_shotgun_subset['V4'] = V4_staphylococcus_df_shotgun_subset[' g__Staphyloccocus V4'].rank(method='min').astype(int)
V4_staphylococcus_df_shotgun_subset

,g__Staphyloccocus V4,V4
LAMI.RD304.D0.C1,0.596710,17
LAMI.RD304.D0.C3,0.806845,22
LAMI.RD304.D11.C1,0.468559,13
LAMI.RD306.D23.C1,0.291059,7
LAMI.RD308.D0.C2,0.198196,3
LAMI.RD306.D11.C1,0.163173,2
LAMI.RD308.D0.C3,0.430618,12
LAMI.RD306.D14.C1,0.273017,5
LAMI.RD304.D14.C3,0.537012,15
LAMI.RD308.D14.C3,0.288405,6


In [ ]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
shotgun_column = staphylococcus_only['Shotgun']
v1_v3_column = V1_V3_staphylococcus_df_shotgun_subset['V1-V3']
v4_column = V4_staphylococcus_df_shotgun_subset['V4']

# Concatenate the selected columns
combined_staphylococcus_df = pd.concat([v1_v3_column, v4_column, shotgun_column], axis=1)

# Rename columns for clarity (optional)
# combined_staphylococcus_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_staphylococcus_df = combined_staphylococcus_df.dropna()

combined_staphylococcus_df

,V1-V3,V4,Shotgun
LAMI.RD304.D14.C3,19,15.0,14
LAMI.RD304.D7.C1,20,16.0,9
LAMI.RD304.D0.C1,22,17.0,6
LAMI.RD306.D11.C1,16,2.0,2
LAMI.RD308.D0.C2,1,3.0,22
LAMI.RD306.D14.C1,21,5.0,1
LAMI.RD304.D0.C3,23,22.0,8
LAMI.RD308.D0.C3,4,12.0,23
LAMI.RD306.D23.C1,12,7.0,15
LAMI.RD306.D14.C3,24,9.0,3


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import pearsonr, norm
import numpy as np

# Extract columns
x = combined_staphylococcus_df['V1-V3']
y = combined_staphylococcus_df['V4']
z = combined_staphylococcus_df['Shotgun']

# Compute pairwise Pearson correlations with p-values
r1, p1 = pearsonr(x, y)
r2, p2 = pearsonr(x, z)
r3, p3 = pearsonr(y, z)

# Compute average r
r_avg = np.mean([r1, r2, r3])

# --- Estimate combined p-value using Fisher's Z transformation ---
def fisher_z(r):
    return np.arctanh(r)

def inverse_fisher_z(z):
    return np.tanh(z)

# Convert to Fisher Z values
z1 = fisher_z(r1)
z2 = fisher_z(r2)
z3 = fisher_z(r3)

# Mean of z-scores
z_avg = np.mean([z1, z2, z3])

# Convert back to average r (already done)
# r_avg = inverse_fisher_z(z_avg)

# Standard error for Fisher z (n ≈ number of samples)
n = len(x)
se = 1 / np.sqrt(n - 3)

# Compute z-statistic and p-value
z_stat = z_avg / se
p_avg = 2 * (1 - norm.cdf(abs(z_stat)))  # two-tailed

# Create 3D scatter plot
fig = plt.figure(figsize=(10, 6))  # Wider for spacing
ax = fig.add_subplot(111, projection='3d')

# Scatter points
scatter = ax.scatter(x, y, z, c='orange', s=60, edgecolors='k')

# Axes labels and title
ax.set_xlabel('16S V1-V3 Rank')
ax.set_ylabel('16S V4 Rank')
ax.set_zlabel('Shotgun metaG Rank')
ax.set_title('Correlation of Staphylococcus by Sequencing Method', fontsize=16)

# Format correlation results with p-values
textstr = '\n'.join((
    f'V1-V3 vs. V4:',
    f'    Pearson r = {r1:.2f}, p = {p1:.3g}',
    '',
    f'V1-V3 vs. Shotgun:',
    f'    Pearson r = {r2:.2f}, p = {p2:.3g}',
    '',
    f'V4 vs. Shotgun:',
    f'    Pearson r = {r3:.2f}, p = {p3:.3g}',
    '',
    f'All averaged:',
    f'    Pearson r = {r_avg:.2f}, p = {p_avg:.3g}',
))

# Annotate text, positioned further left
plt.gcf().text(0.01, 0.35, textstr, fontsize=10)

# Diagonal 1:1:1 correlation line
min_val = min(x.min(), y.min(), z.min())
max_val = max(x.max(), y.max(), z.max())

ax.plot([min_val, max_val], [min_val, max_val], [min_val, max_val],
        color='orange', linewidth=2, linestyle='-', label='1:1:1 Correlation Line')

# Add n value (# of matched samples between all three data types)
textstr = 'n = 22 matched samples'
plt.gcf().text(0.01, 0.21, textstr, fontsize=10)


# Draw dashed lines from each point to the 1:1:1 diagonal
def project_to_diag(x, y, z):
    direction = np.array([1, 1, 1]) / np.sqrt(3)
    point = np.array([x, y, z])
    proj_length = np.dot(point, direction)
    return proj_length * direction

for xi, yi, zi in zip(x, y, z):
    x_proj, y_proj, z_proj = project_to_diag(xi, yi, zi)
    ax.plot([xi, x_proj], [yi, y_proj], [zi, z_proj],
            color='gray', linestyle='--', linewidth=0.7)

# Add legend to label the orange line
ax.legend(loc='lower left', bbox_to_anchor=(-0.4, 0.25))

plt.tight_layout()
plt.savefig('test_3d_with_all_corrs_and_avg_pval_staph.png', dpi=600)
